# Bits and Bytes

*Run a cell with ▶ (or Shift+Enter): Sphinx compiles and runs it on your computer. Change the code and run it again: this notebook is your own copy.*

**After this unit you can:**
- Compute how many values n bits can hold, and how many bits a number needs
- Convert sizes between SI units (KB, MB) and binary units (KiB, MiB)
- Tell bits from bytes in speeds and file sizes, and estimate a download time
- Read a number stored in several bytes, in little-endian or big-endian order

**Exercises** in this notebook end with `check(...)` lines: minimal tests that print ✓ when your code is right. Run the next cell once first, so they can use `check`.

In [ ]:
// shared: check(...) prints ✓ or ✗; the exercises below use it
void check(String what, Object actual, Object expected) {
    if (Objects.equals(actual, expected)) {
        IO.println("✓ " + what);
    } else {
        IO.println("✗ " + what + ": expected " + expected + ", got " + actual);
    }
}

## In short

A **bit** is 0 or 1, and a **byte** is 8 bits. Every extra bit doubles the possibilities, so **n bits have 2ⁿ values**, from 0 to 2ⁿ − 1: a byte holds 0 to 255. Turned around, a number needs as many bits as the times you can halve it before reaching 0: 300 needs 9 (and 0 still needs 1 bit).

![Every extra bit doubles the patterns: 2, 4, 8](bit-patterns.svg)

**Every extra bit doubles the possibilities:**

In [ ]:
for (int bits : new int[] {1, 2, 4, 8, 16, 32, 64}) {
    String values = bits < 64 ? String.format("%,d", 1L << bits) : "18,446,744,073,709,551,616";
    IO.println(String.format("%2d bits: %s values", bits, values));
}

Sizes use two kinds of prefixes. **SI prefixes** multiply by 1000: 1 KB = 1000 bytes, 1 MB = 1000² bytes. **Binary prefixes** multiply by 1024: 1 KiB = 1024 bytes, 1 MiB = 1024² bytes. That's why a 1 TB disk shows as 931 "GB" on Windows: the same bytes divided by 1024³.

**The "missing" space on a 1 TB disk:**

In [ ]:
long bytes = 1_000_000_000_000L; // 1 TB, as the box says
IO.println("in GB  (÷ 1000³): " + bytes / 1_000_000_000.0);
IO.println("in GiB (÷ 1024³): " + bytes / (1024.0 * 1024 * 1024));

Speeds are in **bits** per second (Mbps, lower-case b) and files in **bytes** (MB, upper-case B): a 100 Mbps connection moves at most 12.5 MB per second.

A number bigger than 255 takes several bytes, and they can be stored in two orders. **Big-endian** puts the most significant byte first, as we write numbers; **little-endian** puts the least significant first. Intel and ARM processors and many file formats are little-endian; network protocols are big-endian.

### ✏️ Exercise: One byte of a number

Return byte number `index` of an int, counting from 0 at the least significant end: shift it down and keep 8 bits with `& 0xFF`.

In [ ]:
int byteAt(int value, int index) {
    // TODO: shift by 8 × index, then mask
    return 0;
}

void main() {
    check("low byte of 300", byteAt(300, 0), 0x2C);
    check("high byte of 300", byteAt(300, 1), 1);
    check("top byte", byteAt(0x12345678, 3), 0x12);
}

<details>
<summary>Show an answer</summary>

```java
int byteAt(int value, int index) {
    return (value >> (8 * index)) & 0xFF;
}

void main() {
    check("low byte of 300", byteAt(300, 0), 0x2C);
    check("high byte of 300", byteAt(300, 1), 1);
    check("top byte", byteAt(0x12345678, 3), 0x12);
}
```

</details>

| 300 = 0x012C | 1st byte | 2nd byte |
|--------------|----------|----------|
| big-endian | `01` | `2C` |
| little-endian | `2C` | `01` |

#### 🎞️ Byte order, step by step

![The number 300, 0x012C, stored big-endian as 01 2C and little-endian as 2C 01, and read back as 44 + 1 × 256](endianness-steps.svg)

**Both byte orders, from Java:**

In [ ]:
int value = 300;
byte[] big = ByteBuffer.allocate(4).order(ByteOrder.BIG_ENDIAN).putInt(value).array();
byte[] little = ByteBuffer.allocate(4).order(ByteOrder.LITTLE_ENDIAN).putInt(value).array();
IO.println("big-endian:    " + HexFormat.ofDelimiter(" ").formatHex(big));
IO.println("little-endian: " + HexFormat.ofDelimiter(" ").formatHex(little));

### ✏️ Exercise: Read two little-endian bytes

Put two bytes back together, the low one first, as in a file stored little-endian.

In [ ]:
int fromLittleEndian(int low, int high) {
    // TODO: combine the two bytes
    return 0;
}

void main() {
    check("300", fromLittleEndian(0x2C, 0x01), 300);
    check("the largest", fromLittleEndian(255, 255), 65535);
    check("only the high byte", fromLittleEndian(0, 1), 256);
}

<details>
<summary>Show an answer</summary>

```java
int fromLittleEndian(int low, int high) {
    return low | (high << 8);
}

void main() {
    check("300", fromLittleEndian(0x2C, 0x01), 300);
    check("the largest", fromLittleEndian(255, 255), 65535);
    check("only the high byte", fromLittleEndian(0, 1), 256);
}
```

</details>

## Readings

- [Representing Numbers and Letters with Binary](https://www.youtube.com/watch?v=1GSjbWt0c9M) (Crash Course Computer Science #4, 11 min): The first half: how many values 8 bits can hold, why a kilobyte is sometimes 1000 and sometimes 1024 bytes, and what 32-bit and 64-bit mean.
- [Prefixes for binary multiples](https://physics.nist.gov/cuu/Units/binary.html) (NIST (U.S. National Institute of Standards and Technology), 4 min): The table of binary prefixes (Ki, Mi, Gi) next to the SI ones (k, M, G), and why they were created: so a kilobyte can mean exactly 1000 bytes.
- [Little and Big Endian Mystery](https://www.geeksforgeeks.org/dsa/little-and-big-endian-mystery/) (GeeksforGeeks, 5 min): How the same 4-byte number is laid out in memory in each order, and where each order is used: processors are mostly little-endian, network protocols big-endian.

## Check yourself

1. How many values do 10 bits hold, and how many bits does 1000 need?
2. How many bytes are in 2 MiB, and in 2 MB?
3. Which number do the little-endian bytes `E8 03` store?

## Practice

The notebook covered the basics. Now apply them in the unit's challenges, in the Sphinx sidebar, where Submit runs every test, hidden ones included:

- **How Many Bits?** · Easy
- **Storage Units** · Medium
- **Download Time** · Medium
- **Read a BMP Header** · Hard